In [2]:
import os
import pandas as pd

# 1. Load the raw nav_history file
NAV_PATH = os.path.join("..", "data", "raw", "02_nav_history.csv")
df_nav = pd.read_csv(NAV_PATH)

print(f"📋 Initial Rows in NAV History: {len(df_nav)}")
print("-" * 50)

# A. Parse dates to datetime
# dayfirst=True is crucial for Indian financial datasets typically formatted as DD-MM-YYYY
df_nav['date'] = pd.to_datetime(df_nav['date'], format= 'mixed', dayfirst=True)

📋 Initial Rows in NAV History: 46000
--------------------------------------------------


In [3]:
# B. Sort by amfi_code and date
df_nav = df_nav.sort_values(by=['amfi_code', 'date']).reset_index(drop=True)
print("✅ Dates parsed successfully using 'mixed' format strategy.")

✅ Dates parsed successfully using 'mixed' format strategy.


In [4]:
# C. Remove duplicate entries
duplicate_count = df_nav.duplicated(subset=['amfi_code', 'date']).sum()
df_nav = df_nav.drop_duplicates(subset=['amfi_code', 'date'], keep='first')
print(f"🧹 Removed {duplicate_count} duplicate rows.")

🧹 Removed 0 duplicate rows.


In [5]:
print("🔍 Total Null Values per Column:")
print(df_nav.isnull().sum())

🔍 Total Null Values per Column:
amfi_code    0
date         0
nav          0
dtype: int64


In [7]:
# E. Validate NAV > 0
initial_count = len(df_nav)
df_nav = df_nav[df_nav['nav'] > 0]
invalid_nav_count = initial_count - len(df_nav)
print(f"🛡️ Validated NAV > 0: Dropped {invalid_nav_count} rows.")
print("-" * 50)
print(f"🚀 Final Cleaned Rows in NAV History: {len(df_nav)}")

🛡️ Validated NAV > 0: Dropped 0 rows.
--------------------------------------------------
🚀 Final Cleaned Rows in NAV History: 46000


In [ ]:
processed_dir = os.path.join("..", "data", "processed")
processed_file_path = os.path.join(processed_dir, "02_nav_history_clean.csv")
df_nav.to_csv(processed_file_path, index=False)

print(f"\n💾 Cleaned dataset successfully saved to: {processed_file_path}")


💾 Cleaned dataset successfully saved to: ..\data\processed\02_nav_history_clean.csv


In [9]:
import os
import pandas as pd

# 1. Load the raw transaction dataset
TXN_PATH = os.path.join("..", "data", "raw", "08_investor_transactions.csv")
df_txn = pd.read_csv(TXN_PATH)

print(f"📋 Initial Rows in Transactions Dataset: {len(df_txn)}")
print("\n🔍 Step 0: Checking for Null Values before cleaning:")
print(df_txn.isnull().sum())
print("-" * 60)

📋 Initial Rows in Transactions Dataset: 32778

🔍 Step 0: Checking for Null Values before cleaning:
investor_id           0
transaction_date      0
amfi_code             0
transaction_type      0
amount_inr            0
state                 0
city                  0
city_tier             0
age_group             0
gender                0
annual_income_lakh    0
payment_mode          0
kyc_status            0
dtype: int64
------------------------------------------------------------


In [10]:
# A. Fix Date Formats
# Identifying the date column dynamically (handles 'date', 'txn_date', 'transaction_date')
date_col = [col for col in df_txn.columns if 'date' in col.lower()][0]
df_txn[date_col] = pd.to_datetime(df_txn[date_col], format='mixed', dayfirst=True)
print(f"📅 Date column '{date_col}' successfully parsed to uniform datetime format.")

📅 Date column 'transaction_date' successfully parsed to uniform datetime format.


In [12]:
# B. Standardize Transaction Type
type_col = [col for col in df_txn.columns if 'type' in col.lower()][0]
# Convert to string, strip whitespace, and uppercase to make them perfectly uniform
df_txn[type_col] = df_txn[type_col].astype(str).str.strip().str.upper()
print(f"🔄 Transaction types in '{type_col}' standardized. Unique values now: {df_txn[type_col].unique()}")

🔄 Transaction types in 'transaction_type' standardized. Unique values now: ['SIP' 'REDEMPTION' 'LUMPSUM']


In [13]:
# D. Check KYC Status Values
kyc_col = [col for col in df_txn.columns if 'kyc_status' in col.lower()]
if kyc_col:
    kyc_col = kyc_col[0]
    print(f"\n👤 KYC STATUS VALUES DISTRIBUTION ('{kyc_col}'):")
    print(df_txn[kyc_col].value_counts(dropna=False))
else:
    print("\n⚠️ No explicit KYC column automatically detected.")

print("-" * 60)
print(f"🚀 Final Cleaned Rows in Transactions: {len(df_txn)}")


👤 KYC STATUS VALUES DISTRIBUTION ('kyc_status'):
kyc_status
Verified    30146
Pending      2632
Name: count, dtype: int64
------------------------------------------------------------
🚀 Final Cleaned Rows in Transactions: 32778


In [14]:
processed_dir = os.path.join("..", "data", "processed")

processed_file_path = os.path.join(processed_dir, "clean_transactions.csv")
df_txn.to_csv(processed_file_path, index=False)

print(f"\n💾 Cleaned transaction dataset successfully saved to: {processed_file_path}")
print("=" * 60)


💾 Cleaned transaction dataset successfully saved to: ..\data\processed\clean_transactions.csv


In [15]:
import os
import pandas as pd
import numpy as np

# 1. Load the raw fund performance dataset
PERF_PATH = os.path.join("..", "data", "raw", "07_scheme_performance.csv")
df_perf = pd.read_csv(PERF_PATH)

print(f"📋 Initial Rows in Performance Dataset: {len(df_perf)}")
print("-" * 60)

📋 Initial Rows in Performance Dataset: 40
------------------------------------------------------------


In [16]:
# A. Isolate and validate return columns are numeric
return_cols = [col for col in df_perf.columns if 'return' in col.lower()]
print(f"🎯 Target Return Columns: {return_cols}")

initial_row_count = len(df_perf)
for col in return_cols:
    df_perf[col] = pd.to_numeric(df_perf[col], errors='coerce')

# Drop rows where return columns are invalid/NaN
df_perf = df_perf.dropna(subset=return_cols)
print(f"✅ Return values validated as numeric. Dropped {initial_row_count - len(df_perf)} invalid records.")

🎯 Target Return Columns: ['return_1yr_pct', 'return_3yr_pct', 'return_5yr_pct']
✅ Return values validated as numeric. Dropped 0 invalid records.


In [17]:
# B. Check Expense Ratio Range (Strictly between 0.1% and 2.5%)
expense_col = [col for col in df_perf.columns if 'expense' in col.lower()][0]
# Ensure it is numeric first
df_perf[expense_col] = pd.to_numeric(df_perf[expense_col], errors='coerce')

before_expense_filter = len(df_perf)
# Filtering rows strictly within the 0.1% to 2.5% boundary
df_perf = df_perf[(df_perf[expense_col] >= 0.1) & (df_perf[expense_col] <= 2.5)]
print(f"📉 Checked Expense Ratio range [0.1% - 2.5%]: Filtered out {before_expense_filter - len(df_perf)} non-compliant rows.")

📉 Checked Expense Ratio range [0.1% - 2.5%]: Filtered out 0 non-compliant rows.


In [20]:
# C. Flag Negative Sharpe Ratios
sharpe_col = [col for col in df_perf.columns if 'sharpe' in col.lower()][0]
df_perf[sharpe_col] = pd.to_numeric(df_perf[sharpe_col], errors='coerce')

# 1 if negative, 0 if positive or zero
df_perf['is_underperforming_risk_free'] = np.where(df_perf[sharpe_col] < 0, 1, 0)
negative_sharpe_count = df_perf['is_underperforming_risk_free'].sum()
print(f"🚩 Flagged {negative_sharpe_count} funds with negative Sharpe ratios.")

print("-" * 60)
print(f"🚀 Final Cleaned Rows in Performance Catalog: {len(df_perf)}")

🚩 Flagged 0 funds with negative Sharpe ratios.
------------------------------------------------------------
🚀 Final Cleaned Rows in Performance Catalog: 40


In [21]:
processed_dir = os.path.join("..", "data", "processed")

processed_file_path = os.path.join(processed_dir, "clean_performance.csv")
df_perf.to_csv(processed_file_path, index=False)

print(f"\n💾 Cleaned dataset successfully saved to: {processed_file_path}")
print("=" * 60)


💾 Cleaned dataset successfully saved to: ..\data\processed\clean_performance.csv


In [1]:
import os
import pandas as pd

# 1. Define paths
raw_dir = os.path.join("..", "data", "raw")
processed_dir = os.path.join("..", "data", "processed")

# List of files we have ALREADY cleaned (we skip these so we don't overwrite our custom work!)
custom_cleaned_files = ["02_nav_history.csv", "08_investor_transactions.csv", "07_scheme_performance.csv"]

# 2. Get all remaining CSV files in the raw folder
all_raw_files = [f for f in os.listdir(raw_dir) if f.endswith('.csv')]
remaining_files = [f for f in all_raw_files if f not in custom_cleaned_files]

print(f"🔄 Found {len(remaining_files)} baseline datasets for general sanitization.\n")

# 3. Loop through and clean each remaining dataset
for file_name in remaining_files:
    raw_file_path = os.path.join(raw_dir, file_name)
    
    # Load dataset
    df = pd.read_csv(raw_file_path)
    initial_shape = df.shape
    
    # --- OPERATION A: Remove complete row duplicates ---
    df = df.drop_duplicates().reset_index(drop=True)
    
    # --- OPERATION B: Drop rows that are completely empty (all NaN values) ---
    df = df.dropna(how='all')
    
    # --- OPERATION C: Clean whitespace from text columns ---
    text_cols = df.select_dtypes(include=['object']).columns
    for col in text_cols:
        df[col] = df[col].astype(str).str.strip()
    
    final_shape = df.shape
    
    # Log the output metrics for the report
    print(f"📄 Dataset: {file_name}")
    print(f"   • Initial Matrix : {initial_shape[0]} rows, {initial_shape[1]} columns")
    print(f"   • Sanitized Matrix: {final_shape[0]} rows, {final_shape[1]} columns")
    print(f"   • Rows Removed   : {initial_shape[0] - final_shape[0]}")
    
    # --- OPERATION D: Export to Processed Folder ---
    clean_file_name = file_name.replace(".csv", "_clean.csv")
    output_path = os.path.join(processed_dir, clean_file_name)
    df.to_csv(output_path, index=False)
    print(f"   💾 Saved to: {output_path}\n" + "-"*50)

print("🎉 General data sanitization loop complete! All datasets are polished.")

🔄 Found 12 baseline datasets for general sanitization.

📄 Dataset: 01_fund_master.csv
   • Initial Matrix : 40 rows, 15 columns
   • Sanitized Matrix: 40 rows, 15 columns
   • Rows Removed   : 0
   💾 Saved to: ..\data\processed\01_fund_master_clean.csv
--------------------------------------------------
📄 Dataset: 03_aum_by_fund_house.csv
   • Initial Matrix : 90 rows, 5 columns
   • Sanitized Matrix: 90 rows, 5 columns
   • Rows Removed   : 0
   💾 Saved to: ..\data\processed\03_aum_by_fund_house_clean.csv
--------------------------------------------------
📄 Dataset: 04_monthly_sip_inflows.csv
   • Initial Matrix : 48 rows, 6 columns
   • Sanitized Matrix: 48 rows, 6 columns
   • Rows Removed   : 0
   💾 Saved to: ..\data\processed\04_monthly_sip_inflows_clean.csv
--------------------------------------------------
📄 Dataset: 05_category_inflows.csv
   • Initial Matrix : 144 rows, 3 columns
   • Sanitized Matrix: 144 rows, 3 columns
   • Rows Removed   : 0
   💾 Saved to: ..\data\processed

In [18]:
import os

# 1. Define the schema content
schema_content = """-- ====================================================================
-- STAR SCHEMA BLUEPRINT FOR MUTUAL FUND ANALYTICS DB
-- ====================================================================

-- DIMENSION 1:
CREATE TABLE dim_fund (
    amfi_code INTEGER PRIMARY KEY,
    fund_house TEXT NOT NULL,
    category TEXT,
    expen
);

-- DIMENSION 2: 
CREATE TABLE dim_date (
    date_id INTEGER PRIMARY KEY,
    date INTEGER NOT NULL,
    year INTEGER,
    month INTEGER,
    quarter INTEGER,
    is_weekday INTEGER
);

-- FACT 1: 
CREATE TABLE fact_nav (
    amfi_code INTEGER FOREIGN KEY,
    date INTEGER FOREIGN KEY,
    nav REAL NOT NULL,
    daily_return_pct REAL,
    FOREIGN KEY (amfi_code) REFERENCES dim_fund(amfi_code),
    FOREIGN KEY (date) REFERENCES dim_date(date_id)
);

-- FACT 2:
CREATE TABLE fact_transactions (
    tx_id INTEGER PRIMARY KEY,
    inverstor_id INTEGER,
    amfi_code INTEGER,
    date INTEGER,
    amount REAL,
    type TEXT CHECK(type IN ('LUMPSUM', 'REDEMPTION', 'SIP')),
    FOREIGN KEY (amfi_code) REFERENCES dim_fund(amfi_code)
);

-- FACT 3:
CREATE TABLE fact_performance (
    amfi_code INTEGER ,
    return_1yr_pct REAL,
    sharpe_ratio REAL,
    alpha REAL,
    max_drawdown_pct REAL,
    FOREIGN KEY (amfi_code) REFERENCES dim_fund(amfi_code)
);

-- FACT 4: 
CREATE TABLE fact_portfolio (
    amfi_code INTEGER,
    stock_symbol TEXT,
    weight_pct REAL,
    sector TEXT,
    date DATE
    FOREIGN KEY (amfi_code) REFERENCES dim_fund(amfi_code)
);

-- FACT 5:
CREATE TABLE fact_aum (
    fund_house TEXT,
    date DATE,
    aum_crore REAL,
    num_schemes INTEGER
);

-- FACT 6:
CREATE TABLE fact_sip_industry (
    month INTEGER,
    sip_inflow_crore REAL,
    sip_accounts_crore INTEGER
);
"""

# 2. Establish directory path and write down the schema.sql artifact
database_dir = os.path.join("..", "sql")

schema_file_path = os.path.join(database_dir, "schema.sql")

with open(schema_file_path, "w", encoding="utf-8") as file:
    file.write(schema_content)

print(f"💾 Task A Complete: Star Schema written perfectly to text file:\n📍 {schema_file_path}")

💾 Task A Complete: Star Schema written perfectly to text file:
📍 ..\sql\schema.sql


In [19]:
import os
import sqlalchemy as sa
from sqlalchemy import create_engine, text
import pandas as pd

# =====================================================================
# STEP 1: DEFINE TARGET SCHEMAS
# =====================================================================
table_blueprints = {

    'dim_fund': ['amfi_code', 'fund_house', 'category', 'expense_ratio'],

    'dim_date': ['date_id', 'date', 'month', 'year', 'quarter', 'is_weekday'],

    'fact_nav': ['amfi_code', 'date', 'nav', 'daily_return_pct'],

    'fact_transactions': ['tx_id', 'investor_id', 'amfi_code', 'date', 'amount', 'type'],

    'fact_performance': ['amfi_code', 'return_1yr_pct', 'sharpe_ratio', 'alpha', 'max_drawdown_pct'],

    'fact_portfolio': ['amfi_code', 'stock_symbol', 'weight_pct', 'sector', 'date'],

    'fact_aum': ['fund_house', 'date', 'aum_crore', 'num_schemes'],

    'fact_sip_industry': ['month', 'sip_inflow_crore', 'sip_accounts_crore']

}

# =====================================================================
# STEP 2: SET PATHS & RE-INITIALIZE SCHEMA
# =====================================================================
db_path = os.path.join("..", "sql", "bluestock_mf.db")
schema_path = os.path.join("..", "sql", "schema.sql")
processed_dir = os.path.join("..", "data", "processed")

engine = create_engine(sa.URL.create(drivername="sqlite", database=db_path))

with open(schema_path, "r", encoding="utf-8") as file:
    schema_sql = file.read()

print("🔨 Resetting database tables from schema.sql...")
with engine.begin() as connection:
    for statement in schema_sql.split(";"):
        clean_statement = statement.strip()
        if clean_statement and not clean_statement.startswith("--"):
            connection.execute(text(clean_statement))
print("✅ Base Star Schema compiled perfectly.")
print("-" * 75)

# =====================================================================
# STEP 3: SELF-HEALING INGESTION LOOP
# =====================================================================
csv_files = [f for f in os.listdir(processed_dir) if f.endswith('.csv')]
print(f"📊 Scanning {len(csv_files)} CSV files with automatic column filtering...\n")

route_summary = {table: 0 for table in table_blueprints.keys()}

for csv_name in csv_files:
    csv_path = os.path.join(processed_dir, csv_name)
    try:
        # Sneak peek at columns
        df_preview = pd.read_csv(csv_path, nrows=1)
        csv_cols_raw = list(df_preview.columns)
        csv_cols_lower = [c.lower().strip() for c in csv_cols_raw]
        
        best_table = None
        highest_score = 0
        
        # Determine the best match matching logic
        for table_name, target_cols in table_blueprints.items():
            score = len(set(csv_cols_lower).intersection(set(target_cols)))
            if score > highest_score:
                highest_score = score
                best_table = table_name
                
        if best_table and highest_score >= 1:
            # Load the full data
            df_full = pd.read_csv(csv_path)
            
            # Map raw CSV columns to their exact lowercase names to handle casing mismatches
            df_full.columns = [c.lower().strip() for c in df_full.columns]
            
            # 🚨 THE FIX: Keep ONLY columns that are explicitly expected by the destination table!
            expected_cols = table_blueprints[best_table]
            valid_cols_to_keep = [col for col in df_full.columns if col in expected_cols]
            
            # Filter the dataframe
            df_cleaned = df_full[valid_cols_to_keep].copy()
            
            # Clean up dates
            for col in df_cleaned.columns:
                if 'date' in col.lower():
                    df_cleaned[col] = pd.to_datetime(df_cleaned[col], errors='coerce').dt.strftime('%Y-%m-%d')
            
            # Try to push to database
            df_cleaned.to_sql(best_table, con=engine, if_exists='append', index=False)
            print(f"🎯 INGESTED: '{csv_name}' ➡️ Table '{best_table}' (Kept {len(valid_cols_to_keep)} of {len(csv_cols_raw)} columns)")
            route_summary[best_table] += 1
        else:
            print(f"横️ SKIPPED : '{csv_name}' shared no structural matching attributes.")
            
    except Exception as e:
        print(f"❌ Operational Error on '{csv_name}': {e}")

print("\n" + "="*75)
print("🏁 FINAL CLEAN INGESTION REPORT")
print("="*75)
for t_name, count in route_summary.items():
    print(f"📌 Table '{t_name:<18}' loaded successfully from {count} file(s)")
print("="*75)

🔨 Resetting database tables from schema.sql...
✅ Base Star Schema compiled perfectly.
---------------------------------------------------------------------------
📊 Scanning 10 CSV files with automatic column filtering...

🎯 INGESTED: '01_fund_master_clean.csv' ➡️ Table 'dim_fund' (Kept 3 of 15 columns)
🎯 INGESTED: '03_aum_by_fund_house_clean.csv' ➡️ Table 'fact_aum' (Kept 4 of 5 columns)
🎯 INGESTED: '04_monthly_sip_inflows_clean.csv' ➡️ Table 'fact_sip_industry' (Kept 2 of 6 columns)
🎯 INGESTED: '05_category_inflows_clean.csv' ➡️ Table 'dim_fund' (Kept 1 of 3 columns)
🎯 INGESTED: '06_industry_folio_count_clean.csv' ➡️ Table 'dim_date' (Kept 1 of 6 columns)
🎯 INGESTED: '09_portfolio_holdings_clean.csv' ➡️ Table 'fact_portfolio' (Kept 4 of 8 columns)
❌ Operational Error on '10_benchmark_indices_clean.csv': (sqlite3.OperationalError) table dim_date has no column named date
[SQL: INSERT INTO dim_date (date) VALUES (?)]
[parameters: [('2022-01-03',), ('2022-01-04',), ('2022-01-05',), ('2022

In [20]:
import os
import sqlalchemy as sa
from sqlalchemy import create_engine
import pandas as pd

# =====================================================================
# 1. CHOOSE TARGETS HERE
# =====================================================================
TARGET_TABLE = "fact_performance"         
TARGET_CSV = "clean_performance.csv"

# =====================================================================
# 2. SCHEMATIC DATABASE BLUEPRINTS
# =====================================================================
table_blueprints = {
  'dim_fund': ['amfi_code', 'fund_house', 'category', 'expense_ratio'],

    'dim_date': ['date_id', 'date', 'month', 'year', 'quarter', 'is_weekday'],

    'fact_nav': ['amfi_code', 'date', 'nav', 'daily_return_pct'],

    'fact_transactions': ['tx_id', 'investor_id', 'amfi_code', 'date', 'amount', 'type'],

    'fact_performance': ['amfi_code', 'return_1yr_pct', 'sharpe_ratio', 'alpha', 'max_drawdown_pct'],

    'fact_portfolio': ['amfi_code', 'stock_symbol', 'weight_pct', 'sector', 'date'],

    'fact_aum': ['fund_house', 'date', 'aum_crore', 'num_schemes'],

    'fact_sip_industry': ['month', 'sip_inflow_crore', 'sip_accounts_crore']  
}

# =====================================================================
# 3. PATH EXECUTION & STREAMING
# =====================================================================
base_dir = os.path.dirname(os.getcwd())
db_path = os.path.abspath(os.path.join(base_dir, "sql", "bluestock_mf.db"))
csv_path = os.path.abspath(os.path.join(base_dir, "data", "processed", TARGET_CSV))

engine = create_engine(sa.URL.create(drivername="sqlite", database=db_path))

if TARGET_TABLE not in table_blueprints:
    raise ValueError(f"❌ '{TARGET_TABLE}' is not a valid table name in your Star Schema.")

if not os.path.exists(csv_path):
    raise FileNotFoundError(f"❌ File not found at: {csv_path}")

try:
    # Load dataset and force columns to lowercase to avoid casing mismatch issues
    df = pd.read_csv(csv_path)
    df.columns = [c.lower().strip() for c in df.columns]
    
    # Filter: Intercept and keep ONLY columns that this specific database table expects
    expected_cols = table_blueprints[TARGET_TABLE]
    valid_cols_to_keep = [col for col in df.columns if col in expected_cols]
    
    df_cleaned = df[valid_cols_to_keep].copy()
    
    # Standardize dates to YYYY-MM-DD
    for col in df_cleaned.columns:
        if 'date' in col.lower():
            df_cleaned[col] = pd.to_datetime(df_cleaned[col], errors='coerce').dt.strftime('%Y-%m-%d')
            
    # Load into SQLite
    print(f"🚀 Streaming data from '{TARGET_CSV}' into database table '{TARGET_TABLE}'...")
    df_cleaned.to_sql(TARGET_TABLE, con=engine, if_exists='append', index=False)
    
    print(f"✅ SUCCESS: Loaded {len(df_cleaned)} rows into '{TARGET_TABLE}'.")
    print(f"💡 Columns Kept: {valid_cols_to_keep}")

except Exception as e:
    print(f"❌ Ingestion Failed: {e}")

🚀 Streaming data from 'clean_performance.csv' into database table 'fact_performance'...
❌ Ingestion Failed: (sqlite3.OperationalError) table fact_performance has no column named return_1yr_pct
[SQL: INSERT INTO fact_performance (amfi_code, return_1yr_pct, alpha, sharpe_ratio, max_drawdown_pct) VALUES (?, ?, ?, ?, ?)]
[parameters: [(119551, 12.42, 0.87, 0.88, -21.7), (119552, 15.25, 1.78, 0.81, -24.43), (119598, 24.56, 1.23, 0.94, -13.35), (119599, 20.59, 1.13, 0.93, -24.78), (119120, 5.34, 1.6, 1.52, -2.3), (100016, 10.94, 0.78, 1.06, -17.41), (125497, 11.48, 1.13, 0.96, -33.5), (100033, 15.43, 0.95, 0.87, -13.67)  ... displaying 10 of 40 total bound parameter sets ...  (149323, 14.12, 1.02, 0.9, -26.99), (149324, 20.2, 0.69, 0.8, -17.01)]]
(Background on this error at: https://sqlalche.me/e/20/e3q8)
